В начале я опишу то, как создавался датасет. В Readme Вам предоставлены две ссылки на датасеты, собранный мной датасет состоит из них, в train часть попали часть изображений iiit5k и весь датасет из icdar-2015, а в для валидации использовался iiit5k. Выбор именно двух датасетом был осознанным, так как на первых порах моя модель переобучалась на данных только из iiit5k, плюс данных не хватало(всего было порядка 2000 обычных изображений и 2000 перевёрнутых, однако на тесте у нас 20000).  

В коде ниже я беру все картинки из обоих датасетов и переворачиваю их, чтобы модель могла обучаться, так как идеального датасета конкретно под эту задачу я не нашёл, также я создаю csv файлы с названием каждого изображения и меткой 0 или 1, если изображение не перевёрнуто или перевёрнуто соответственно

In [ ]:
import os
from pathlib import Path
from PIL import Image
import shutil
import csv

import random, numpy as np, torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed(42) # фиксируем random_state для воспроизводимости

train_data = ['/kaggle/input/datasets/prathmeshzade/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/train',
             '/kaggle/input/datasets/dattrinh12/icdar-2015/Task 4.3/Test',
             '/kaggle/input/datasets/dattrinh12/icdar-2015/Task 4.3/Train'] # ссылки на используемые мной открытые датасеты
train_csv = Path('/kaggle/working/train_labels.csv')
test_csv = Path('/kaggle/working/test_labels.csv') # пути к csv файлам, где мы запишем метки для каждого изображения
with open(train_csv, mode='w', newline='', encoding='utf-8') as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(['filename', 'label'])
    for train_path in train_data:
        cnt = 0
        output_dir_train = Path('/kaggle/working/train_rotated') # создадим директорию с перевёрнутыми изображениями
        output_dir_train.mkdir(exist_ok=True)
        for item in os.listdir(train_path):
            img_path = f"{train_path}/{item}"
            item, ext = item.split('.') # отделим название файла от расширения
            if ext != 'png': continue
            im = Image.open(img_path)
            out = im.rotate(180)
            rotated_img = f"{output_dir_train}/{item}_rotated.{ext}"
            out.save(rotated_img) # сохраняем перевёрнутые изображения
            orig_img = f"{output_dir_train}/{item}.{ext}"
            shutil.copy2(img_path, orig_img) # сохраняем и новые и старые изображения в working директорию, 
            # этого можно было не делать, но таким образом, мы по сути создаём собственный датасет
            writer.writerow([f'{item}.{ext}', 0])
            writer.writerow([f'{item}_rotated.{ext}', 1])

    

# аналогично развернём все тестовые изображения
test_path = '/kaggle/input/datasets/prathmeshzade/iiit5k-words/IIIT5K-Word_V3.0/IIIT5K/test'
output_dir_test = Path('/kaggle/working/test_rotated')
output_dir_test.mkdir(exist_ok=True)
with open(test_csv, mode='w', newline='', encoding='utf-8') as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow(['filename', 'label'])
    for item in os.listdir(test_path):
        img_path = f"{test_path}/{item}"
        item, ext = item.split('.') # отделим название файла от расширения
        im = Image.open(img_path)
        out = im.rotate(180)
        rotated_img = f"{output_dir_test}/{item}_rotated.{ext}"
        out.save(rotated_img)
        orig_img = f"{output_dir_test}/{item}.{ext}"
        shutil.copy2(img_path, orig_img)
        writer.writerow([f'{item}.{ext}', 0])
        writer.writerow([f'{item}_rotated.{ext}', 1])

In [ ]:
import os
import pandas as pd
import cv2

freq_size = dict() # как часто какой размер встречается
sum_width = 0
sum_height = 0
cnt = 0
for image in os.listdir(output_dir_train):
  img = cv2.imread(f"{output_dir_train}/{image}")
  width, height, color = img.shape
  cnt += 1
  sum_width += width
  sum_height += height
  freq_size[f"{width} * {height}"] = freq_size.get(f"{width} * {height}", 0) + 1

print(f"Средняя ширина: {sum_width / cnt}. Средняя высота: {sum_height / cnt}")
pd.DataFrame.from_dict(freq_size, orient='index') # здесь я проверил на train какие вообще размеры у изображений, выяснилось, что они все довольно разные
# в связи с чем resize я буду делать стандартный 64 * 256

Обучим ResNet50 на нашем датасете предсказывать вероятности. На ранних сабмитах я обучал ResNet18, но ResNet50 потяжелее и способен выявлять большие закономерности, при этом время инференса модели сильно не пострадает

Для начала создадим dataloader для обучения

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path

train_dir = Path('/kaggle/working/train_rotated')
train_csv = Path('/kaggle/working/train_labels.csv')
test_dir = Path('/kaggle/working/test_rotated')
test_csv = Path('/kaggle/working/test_labels.csv')

class TextOrientationDataset(Dataset): # это класс датасета, который мы будем использовать в DataLoader
    def __init__(self, image_dir, csv_path, transform=None):
        self.image_dir = Path(image_dir)
        self.transform = transform
        self.df = pd.read_csv(csv_path)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx): # принимаем индекс и возвращаем картинку и метку перевёрнута она или нет
        row = self.df.iloc[idx]
        img_path = self.image_dir / row['filename']
        label = row['label']
        
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        
        return img, torch.tensor([label], dtype=torch.float32)


train_transform = transforms.Compose([ # здесь мы преобразовываем каждое изображение из тестовой выборки, при этом важно, что все аугментации мы применяем только к train
    # на ранних сабмитах аугментаций было меньше, здесь уже добавлены более сильные варианты по типу случайного обрезания и искажения.
    transforms.Resize((64, 256)),
    
    # Размытие
    transforms.RandomApply([
        transforms.GaussianBlur(kernel_size=5, sigma=(0.5, 3.0))
    ], p=0.7),
    
    # Изменение цвета
    transforms.ColorJitter(
        brightness=0.5,
        contrast=0.5,
        saturation=0.3,
        hue=0.1
    ),
    
    # Геометрические искажения
    transforms.RandomAffine(
        degrees=10,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1)
    ),
    
    # Случайное обрезание
    transforms.RandomApply([
        transforms.RandomResizedCrop((64, 256), scale=(0.85, 1.0))
    ], p=0.4),
    
    # Нормализация
    transforms.ToTensor(),
    # Случайное стирание частей
    transforms.RandomErasing(
        p=0.3,
        scale=(0.02, 0.15),
        ratio=(0.3, 3.3),
        value='random'
    ),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((64, 256)),
    transforms.ToTensor(), # стандартные трансформации для валидационных изображений
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = TextOrientationDataset(train_dir, train_csv, train_transform)
val_dataset = TextOrientationDataset(test_dir, test_csv, val_transform)


train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=2)

Теперь обучим предобученную на ImageNet ResNet18

Лосс при обучении я буду считать не при помощи Brier Score, а при помощи accuracy, чтобы оценить насколько модель в целом понимает, где перевёрнутые изображения, а где нет. Однако на тесте я буду использовать Brier Score для более точной оценки качества и понимания слабых мест нашей модели

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, 1)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss() # в виде лосса возьмём бинарную кросс энтропию
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

num_epochs = 20
best_val_loss = float('inf')

for epoch in range(num_epochs):
    # Обучение
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    
    train_loss /= len(train_loader)
    train_acc = correct / total
    
    # Валидация
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            preds = (torch.sigmoid(outputs) >= 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    val_loss /= len(val_loader)
    val_acc = correct / total
    
    print(f"Epoch {epoch + 1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
    
    # Сохраняем лучшую модель
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
    
    scheduler.step()

print(f"Лучший validation loss:{best_val_loss}")
print(f"Лучший validation accuracy:{best_val_acc}")

Соберём все логиты с валидации, чтобы подобрать оптимальную температуру. Основная идея в том, что при Brier score наша модель не должна быть слишком уверена или неуверена в своих ответах, так как в случае полной уверенности мы получим (1 - 0.99)^2, что является маленькой величиной, но в случае ошибки, если модель была уверена в том, что изображение перевёрнуто, мы получим (0.99 - 0)^2, что даёт большой штраф, поэтому имеет смысл делать модель менее уверенной для подстраховки от такого рода ошибок

In [ ]:
import torch
import numpy as np
from scipy.optimize import minimize

# загружаем модель
model.load_state_dict(torch.load('/kaggle/working/best_model.pth'))
model.eval()

# собираем все метки и логиты с валидации
val_logits = []
val_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        logits = model(images)
        
        val_logits.append(logits.cpu().numpy())
        val_labels.append(labels.cpu().numpy())

# Объединяем все батчи в один массив
val_logits = np.concatenate(val_logits)
val_labels = np.concatenate(val_labels)

Найдём оптимальную температуру и затем применим её к тестовым данным

In [ ]:
# Функция Brier Score с температурой
def brier_score_with_temperature(T, logits, targets):
    # Применяем температуру и сигмоиду
    calibrated_probs = 1 / (1 + np.exp(-logits / T))
    # Считаем Brier Score
    return np.mean((calibrated_probs - targets) ** 2)

result = minimize(
    brier_score_with_temperature,
    x0=[1.0],  # Начальное приближение для T
    args=(val_logits, val_labels),
    bounds=[(0.1, 10.0)],  # Ищем T в диапазоне [0.1, 10.0]
    method='L-BFGS-B'  # Метод оптимизации
)

optimal_T = result.x[0]
print(f"Оптимальная температура: {optimal_T:.4f}")

# Сохраняем температуру
with open('/kaggle/working/temperature.txt', 'w') as f:
    f.write(str(optimal_T))

In [ ]:
test_transform = transforms.Compose([
    transforms.Resize((64, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class TestDataset(Dataset): # аналогичный датасет только для теста
    def __init__(self, image_dir, transform=None):
        self.image_dir = Path(image_dir)
        self.transform = transform
        # Собираем все картинки и сортируем по имени
        self.files = sorted([
            f for f in self.image_dir.iterdir() 
            if f.suffix.lower() in ['.jpg', '.jpeg', '.png']
        ])
        print(f" Найдено {len(self.files)} тестовых изображений")
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        img_path = self.files[idx]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        # Возвращаем картинку и имя файла без расширения (для submission)
        return img, img_path.stem  # stem = имя без расширения

test_dir = Path("/kaggle/input/datasets/balandinbogdan/test-dataset/test/images")
test_dataset = TestDataset(test_dir, test_transform)
test_loader = DataLoader(
    test_dataset, 
    batch_size=128,
    shuffle=False,
    num_workers=2
)

Загрузка и инференс модели

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import pandas as pd
import numpy as np
from pathlib import Path
import time

# загрузка модели
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 1)
model.load_state_dict(torch.load('/kaggle/working/best_model.pth'))
model = model.to(device)
model.eval()

# инференс модели

# будем также замерять время инференса модели
start_time = time.time() 
predictions = []

with torch.no_grad():
    for images, image_ids in test_loader:
        images = images.to(device)
        logits = model(images)
        logits_np = logits.cpu().numpy()
        calibrated_probs = 1 / (1 + np.exp(-logits_np / optimal_T))
        for image_id, prob in zip(image_ids, calibrated_probs):
            predictions.append({
                'image_id': image_id,
                'p_180': float(prob[0])
            })
            
end_time = time.time()  # ← Засекаем время ПОСЛЕ
total_time = end_time - start_time

total_images = len(predictions)
images_per_second = total_images / total_time
print(f"Время инференса на всём датасете (20000 изображений): {total_time}")
print(f"Количество изображений за 1 секунду: {images_per_second}")

# сохраняем итог в submission.csv
submission = pd.DataFrame(predictions)
submission.to_csv('/kaggle/working/submission.csv', index=False)